In [0]:
!pip install statsforecast

In [0]:
import pyspark.sql.functions as F
from statsforecast import StatsForecast
from statsforecast.models import Naive,SeasonalNaive,WindowAverage
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_percentage_error

In [0]:
fe_data_path = "/Volumes/workspace/m5/processed/FE"

In [0]:
data = pd.read_parquet(fe_data_path)
prod_cus_dim = data[['id','item_id','dept_id','cat_id','store_id','state_id','segment']].drop_duplicates().rename(columns={'id':"unique_id"})

In [0]:
data = data[['id','yearweek','sales']].drop_duplicates()
data = data.sort_values(by=['id','yearweek'])
data["date"] = pd.to_datetime(data['yearweek'].astype(str) + "1", format="%G%V%u")
data = data[data['yearweek']!="201653"]
data.head()

In [0]:
df = data[['id','date','sales']].rename(columns = {'id':'unique_id','date':'ds','sales':'y'})
df.head()

In [0]:
df.agg(min_yw = ('ds','min'),max_yw=('ds','max')).display()

In [0]:
train_df = df.groupby('unique_id',as_index=False).apply(lambda df:df.iloc[:-20,:])
val_df = df.groupby('unique_id',as_index=False).apply(lambda df:df.iloc[-20:,:])

In [0]:
baseline_models = [
    Naive(),
    SeasonalNaive(season_length=52),
    WindowAverage(window_size=4)
]

In [0]:
train_df['unique_id'].nunique()

In [0]:
sf = StatsForecast(
    models = baseline_models,
    freq = 'W-MON',
)

y_pred = sf.forecast(df=train_df,h=20)

In [0]:
validation_df = y_pred.merge(val_df,on=['unique_id','ds'],how='left').merge(prod_cus_dim,on=['unique_id'],how='left')
validation_df.head()

In [0]:
state_cat_df = validation_df.groupby(['cat_id'],as_index=False).agg(naive_pred = ('Naive','sum'),
                                                                seasonal_pred = ('SeasonalNaive','sum'),
                                                                windowAvg = ('WindowAverage','sum'),
                                                                y_true = ('y','sum'),
                                                                item_count = ('unique_id','nunique'))

state_cat_df['naive_mape'] = (
    np.round(abs(state_cat_df['y_true'] - state_cat_df['naive_pred'])
    / state_cat_df['y_true'],2)
    * 100
)

state_cat_df['seasonal_mape'] = (
    np.round(abs(state_cat_df['y_true'] - state_cat_df['seasonal_pred'])
    / state_cat_df['y_true'],2)
    * 100
)

state_cat_df['windowAvg_mape'] = (
    np.round(abs(state_cat_df['y_true'] - state_cat_df['windowAvg'])
    / state_cat_df['y_true'],2)
    * 100
)

state_cat_df.display()

In [0]:
segment_df = validation_df.groupby(['segment'],as_index=False).agg(naive_pred = ('Naive','sum'),
                                                                seasonal_pred = ('SeasonalNaive','sum'),
                                                                windowAvg = ('WindowAverage','sum'),
                                                                y_true = ('y','sum'),
                                                                item_count = ('unique_id','nunique'))

segment_df['naive_mape'] = (
    np.round(abs(segment_df['y_true'] - segment_df['naive_pred'])
    / segment_df['y_true'],2)
    * 100
)

segment_df['seasonal_mape'] = (
    np.round(abs(segment_df['y_true'] - segment_df['seasonal_pred'])
    / segment_df['y_true'],2)
    * 100
)

segment_df['windowAvg_mape'] = (
    np.round(abs(segment_df['y_true'] - segment_df['windowAvg'])
    / segment_df['y_true'],2)
    * 100
)

segment_df.display()

In [0]:
validation_df.groupby(['segment','unique_id']).agg(sales_sum=('y','sum')).sort_values(by=['sales_sum'],ascending=False).reset_index().display()

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred,
        unique_ids=['FOODS_3_090_CA_3','FOODS_3_586_CA_3','FOODS_3_120_CA_3','FOODS_3_090_CA_1'],
        models = ["Naive",'SeasonalNaive','WindowAverage'])

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred,
        unique_ids = ['FOODS_3_329_CA_3','FOODS_3_808_CA_3','FOODS_3_086_CA_3','FOODS_3_064_CA_2'],
        models = ["Naive",'SeasonalNaive','WindowAverage']
        )

In [0]:


sf.plot(df=df,
        forecasts_df=y_pred,
        unique_ids = ['FOODS_3_587_CA_1','FOODS_3_511_CA_1','FOODS_3_120_CA_2','FOODS_3_587_CA_2'],
        models = ["Naive",'SeasonalNaive','WindowAverage']
        )

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred,
        unique_ids = ['FOODS_3_572_CA_3','FOODS_3_135_CA_1','HOUSEHOLD_1_094_CA_3'],
        models = ["Naive",'SeasonalNaive','WindowAverage']
        )